# 04.2 · Modelagem — o cotovelo com *label encoding* e com o preço

**CRISP-DM fase 4 · modelagem** · Meridiano Imobiliária

---

Este notebook repete o [04.1](04-1_modelagem_cotovelo.ipynb) — os mesmos oito algoritmos, a
mesma varredura e o mesmo gráfico de cotovelo ao vivo — com **duas mudanças no espaço** em que
os imóveis são comparados:

1. **as nominais entram por *label encoding*, e não por *one-hot*.** Cada categoria vira um
   número inteiro (0, 1, 2…), e a coluna continua sendo **uma** coluna. No 04.1, as 20
   nominais viravam ~150 colunas 0/1 e dominavam a distância;
2. **o preço de venda entra na distância.** No 04.1 ele ficava de fora, reservado para
   descrever os segmentos depois.

A pergunta é se a silhueta melhora com isso. O bloco 7 isola cada efeito: compara o espaço
do 04.1, o *label encoding* sem o preço e o *label encoding* com o preço, com os mesmos
algoritmos e a mesma semente.

Cada algoritmo é treinado **para cada número de grupos**, e o leitor acompanha, em tempo de
execução, duas curvas lado a lado:

* **o peso** — o custo que o algoritmo minimiza (soma das distâncias, inércia,
  −log-verossimilhança). Ele **sempre cai** quando *k* aumenta: mais grupos, cada ponto mais
  perto do seu centro. O **cotovelo** é o ponto a partir do qual um grupo a mais deixa de
  comprar redução relevante de peso;
* **a silhueta** — a métrica de acompanhamento comum a todos. Vai de −1 a 1 e mede, para cada
  imóvel, se ele está mais perto do próprio grupo do que do grupo vizinho. Ao contrário do
  peso, ela **não melhora sozinha** com *k*, e por isso serve de contraprova ao cotovelo.

Os algoritmos treinados:

| # | Algoritmo | O que é *k* | Peso (o que cai com *k*) | Silhueta medida em |
|---|---|---|---|---|
| 1 | K-Means · euclidiana | número de centroides | inércia (soma dos quadrados) | distância euclidiana |
| 2 | K-Means · Manhattan | número de centros | soma das distâncias L1 | distância Manhattan |
| 3 | K-Means · Mahalanobis | número de centroides | soma dos quadrados de Mahalanobis | distância de Mahalanobis |
| 4 | K-Medoids | número de medoides | soma das distâncias ao medoide | distância euclidiana |
| 5 | Mistura gaussiana (GMM) | número de componentes | −log-verossimilhança | distância euclidiana |
| 6 | Hierárquica (Ward) | altura do corte no dendrograma | soma dos quadrados intra-grupo | distância euclidiana |
| 7 | Mean Shift | **emerge** da largura de banda | soma dos quadrados intra-grupo | distância euclidiana |
| 8 | DBSCAN | **emerge** do raio `eps` | soma dos quadrados, ruído sem grupo | distância euclidiana, sem ruído |

Em Mean Shift e DBSCAN o número de grupos não é parâmetro. Neles a varredura percorre o
parâmetro que **controla** *k* (do maior para o menor), e cada ponto do gráfico traz o *k*
que resultou.

Cada silhueta é medida na distância que o próprio algoritmo usa. Para comparar algoritmos
entre si, a tabela final traz também a **silhueta comum**: a euclidiana, no mesmo espaço,
para todos.

## 0 · Ambiente

In [ ]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.spatial.distance import cdist, pdist, squareform
from sklearn.cluster import DBSCAN, KMeans, MeanShift, estimate_bandwidth
from sklearn.compose import ColumnTransformer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.model import densidade as den
from src.model import hierarquica as hier
from src.model import particional as part
from src.utils import io

io.configurar_estilo()

SEMENTE = config.SEMENTE

# Varredura de k: começa em 1 porque o peso com um único grupo é a âncora da curva
# (a dispersão total da carteira). A silhueta só existe a partir de k = 2.
K_VALORES = list(range(1, 11))

# A faixa que o negócio consegue operar como linha de produto (config do projeto).
FAIXA_NEGOCIO = (config.K_MINIMO_NEGOCIO, config.K_MAXIMO_NEGOCIO)

## 1 · Dados e pré-processamento — *label encoding* nas nominais, preço na distância

In [ ]:
data_path = Path("../data/processed/imoveis_curados.parquet")
dict_path = Path("../data/processed/dicionario.csv")

df = pd.read_parquet(data_path)
df_dict = pd.read_csv(dict_path, sep=";").assign(coluna=lambda d: d["variavel"])

display(Markdown(
    f"### Conjunto de dados — {len(df):,} imóveis × {df.shape[1]} colunas".replace(",", ".")
))
display(df.head())

In [ ]:
# Além dos identificadores, sai `subclasse`: é um código do tipo de imóvel (20, 60, 190…)
# que o dicionário declara "Discreta". Normalizado, vira uma distância entre códigos.
#
# `preco_venda` FICA: é a pergunta deste notebook. O dicionário o declara "Discreta", então
# ele entra pelo filtro de tipo, junto das contínuas.
useless_columns = ["ordem", "pid", "subclasse"]


def colunas_do_tipo(*tipos):
    # Colunas da base curada que o dicionário declara com um dos `tipos`.
    return [
        coluna
        for coluna in df_dict.query("Tipo in @tipos")["coluna"].dropna()
        if coluna in df.columns and coluna not in useless_columns
    ]


# Contínua e discreta, mais as derivadas da curadoria
derived_continuous = ["area_varandas", "banheiros_total", "idade_imovel", "anos_desde_reforma"]
derived_binary = ["tem_porao", "tem_garagem", "tem_lareira", "tem_segundo_pavimento"]
continuous_columns = colunas_do_tipo("Contínua", "Discreta") + derived_continuous + derived_binary
assert "preco_venda" in continuous_columns, "o preço precisa estar na distância"

# Ordinal & Nominal
ordinal_columns = colunas_do_tipo("Ordinal")
nominal_columns = colunas_do_tipo("Nominal")

display(Markdown(
    f"- contínuas e discretas: **{len(continuous_columns)}**\n"
    f"- ordinais: **{len(ordinal_columns)}**\n"
    f"- nominais: **{len(nominal_columns)}**\n"
    f"- fora da modelagem: **{len(useless_columns)}** {useless_columns}"
))

### *Label encoding*, e o que ele supõe

No `scikit-learn`, o `LabelEncoder` é feito para o rótulo `y` e codifica uma coluna por vez.
Para as colunas de `X`, o equivalente é o `OrdinalEncoder`: a mesma codificação — as
categorias em ordem alfabética, numeradas de 0 a *n*−1 —, aplicada às 20 nominais de uma vez
e reaproveitável dentro de um `Pipeline`. Depois dele, o `MinMaxScaler` leva cada código para
o intervalo [0, 1], a mesma escala das outras colunas.

O ganho é de dimensão: cada nominal volta a pesar como **uma** variável, e não como até 28.
O custo é uma **ordem que não existe**. Para o *label encoding*, `Old Town` (código 20) está
mais perto de `Somerset` (23) do que de `Brookside` (3) — pela letra inicial, não por
qualquer semelhança entre os bairros. A célula abaixo mostra os códigos do bairro para que
isso fique visível. Com 28 bairros, a "distância" entre dois deles vai de 0 a 1 conforme o
alfabeto.

Duas consequências para ler o resultado:

* uma silhueta maior aqui **não** quer dizer que os grupos ficaram mais parecidos por dentro
  — quer dizer que a régua mudou. Comparar silhuetas entre o 04.1 e o 04.2 é comparar
  réguas; o bloco 7 faz isso de forma explícita;
* com o preço na distância, descrever os segmentos pelo preço passa a ser **circular**: o
  preço ajudou a formar os grupos que depois diferencia.

In [ ]:
def montar_preprocessador(nominais_como: str, com_preco: bool) -> ColumnTransformer:
    # O mesmo ColumnTransformer do 07, com a codificação das nominais e o preço como
    # parâmetros. O bloco 7 usa as outras combinações para isolar cada efeito.
    continuas = [c for c in continuous_columns if com_preco or c != "preco_venda"]
    if nominais_como == "label":
        nominal = Pipeline([
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ("normalization", MinMaxScaler()),
        ])
    else:
        nominal = Pipeline([
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
    return ColumnTransformer(transformers=[
        ("continuous", Pipeline([("normalization", MinMaxScaler())]), continuas),
        ("ordinal", Pipeline([("normalization", MinMaxScaler())]), ordinal_columns),
        ("nominal", nominal, nominal_columns),
    ])


preprocessor = montar_preprocessador("label", com_preco=True)

X = df.drop(columns=useless_columns)  # aprendizagem não supervisionada: não há y
X_T = preprocessor.fit_transform(X)
N = len(X_T)

display(Markdown(
    f"Espaço pré-processado: **{N:,} × {X_T.shape[1]}**".replace(",", ".")
    + " — no 04.1 eram 208 colunas, a maior parte delas *one-hot*"
))

# Os códigos que o label encoding deu ao bairro: ordem alfabética, sem significado.
codificador = preprocessor.named_transformers_["nominal"].named_steps["encoder"]
bairros = codificador.categories_[nominal_columns.index("bairro")]
display(Markdown(
    f"Códigos do `bairro` ({len(bairros)} categorias, em ordem alfabética): "
    + " · ".join(f"`{i}` {b}" for i, b in enumerate(bairros))
))

## 2 · Os três espaços de distância

A silhueta e o K-Medoids trabalham sobre distâncias entre pares de imóveis. Elas são
calculadas **uma vez** e reaproveitadas em toda varredura — é o que permite medir a
silhueta a cada *k* sem que o gráfico demore a andar.

**Mahalanobis.** A distância de Mahalanobis desconta a correlação entre as variáveis:
$d(x, y) = \sqrt{(x-y)^\top \Sigma^{-1} (x-y)}$. Ela é exatamente a distância euclidiana
depois de **branquear** os dados — rotacionar para os componentes principais e dividir cada
um pelo próprio desvio-padrão. Rodar o K-Means euclidiano no espaço branqueado **é** rodar o
K-Means com Mahalanobis, e a célula confere isso numericamente.

Sem o *one-hot*, $\Sigma$ continua singular: `condicao_venda` é constante (a curadoria
manteve só as vendas normais) e há derivadas que repetem a informação de outras colunas. As
direções de variância nula ficam fora (é a pseudo-inversa), e isso também é conferido.

In [ ]:
# Euclidiana e Manhattan, no espaço pré-processado.
D_EUCLIDIANA = squareform(pdist(X_T, "euclidean"))
D_MANHATTAN = squareform(pdist(X_T, "cityblock"))

# Branqueamento: Z = (X - μ) V Λ^(-1/2), só nas direções com variância > 0.
covariancia = np.cov(X_T, rowvar=False)
autovalores, autovetores = np.linalg.eigh(covariancia)
mantidos = autovalores > 1e-10 * autovalores.max()
Z_MAHALANOBIS = (X_T - X_T.mean(axis=0)) @ autovetores[:, mantidos] / np.sqrt(autovalores[mantidos])
D_MAHALANOBIS = squareform(pdist(Z_MAHALANOBIS, "euclidean"))

# A prova: Mahalanobis com a pseudo-inversa = euclidiana no espaço branqueado.
amostra = slice(0, 200)
referencia = cdist(X_T[amostra], X_T[amostra], "mahalanobis", VI=np.linalg.pinv(covariancia))
erro = np.abs(referencia - D_MAHALANOBIS[amostra, amostra]).max()
assert erro < 1e-8, f"o branqueamento não reproduz Mahalanobis (erro {erro:.2e})"

display(Markdown(
    f"- direções com variância nula descartadas no branqueamento: "
    f"**{(~mantidos).sum()}** de {len(mantidos)}\n"
    f"- erro máximo entre Mahalanobis (`scipy`) e euclidiana branqueada: **{erro:.1e}**"
))

## 3 · O gráfico ao vivo

Cada varredura abre um painel com dois gráficos e o **redesenha a cada *k* treinado**: o
peso à esquerda, a silhueta à direita. Quando a varredura termina, o painel marca:

* a **linha tracejada** no cotovelo — o *k* mais distante da corda que liga o primeiro ao
  último ponto da curva de peso (`part.cotovelo`, o mesmo critério do notebook 04);
* a **estrela** no *k* de maior silhueta;
* a **faixa sombreada** de 2 a 5 grupos, o que o negócio opera;
* a **linha pontilhada** na silhueta mínima aceita pelo projeto.

Quando cotovelo e estrela coincidem, as duas leituras concordam. Quando divergem, o peso
está premiando grupos que a silhueta não reconhece como separados — e vale desconfiar.

In [ ]:
RESULTADOS: dict[str, dict] = {}


def contar_grupos(rotulos: np.ndarray) -> int:
    # Grupos válidos: o rótulo -1 é ruído (DBSCAN) e não conta como grupo.
    return len(np.unique(rotulos[rotulos >= 0]))


def soma_quadrados_intra(dados: np.ndarray, rotulos: np.ndarray) -> float:
    # Soma das distâncias quadráticas de cada ponto ao centroide do próprio grupo.
    return float(sum(
        ((dados[rotulos == g] - dados[rotulos == g].mean(axis=0)) ** 2).sum()
        for g in np.unique(rotulos)
    ))


def silhueta_sobre(distancias: np.ndarray):
    # Devolve uma função que mede a silhueta numa matriz de distâncias pré-calculada.
    # Ruído fica de fora; com menos de dois grupos, a silhueta não existe (NaN).
    def medir(rotulos: np.ndarray) -> float:
        validos = rotulos >= 0
        grupos = contar_grupos(rotulos)
        if grupos < 2 or validos.sum() <= grupos:
            return np.nan
        submatriz = distancias[np.ix_(validos, validos)]
        return float(silhouette_score(submatriz, rotulos[validos], metric="precomputed"))

    return medir


silhueta_comum = silhueta_sobre(D_EUCLIDIANA)


def cotovelo_continuo(curva: pd.Series):
    # O critério de `part.cotovelo` (maior distância à corda) para um eixo contínuo,
    # como bandwidth e eps. Devolve o valor do parâmetro, não um k.
    x = curva.index.to_numpy(dtype=float)
    y = curva.to_numpy(dtype=float)
    pontos = np.column_stack([(x - x.min()) / (x.max() - x.min()),
                              (y - y.min()) / (y.max() - y.min())])
    inicio, fim = pontos[0], pontos[-1]
    vetor = (fim - inicio) / np.linalg.norm(fim - inicio)
    relativos = pontos - inicio
    distancias = np.linalg.norm(relativos - np.outer(relativos @ vetor, vetor), axis=1)
    return curva.index[int(np.argmax(distancias))]


class CotoveloAoVivo:
    # Painel peso × silhueta, redesenhado a cada ponto da varredura.

    def __init__(self, nome, rotulo_peso, rotulo_silhueta, rotulo_x, por_parametro):
        self.nome = nome
        self.rotulo_peso = rotulo_peso
        self.rotulo_silhueta = rotulo_silhueta
        self.rotulo_x = rotulo_x
        self.por_parametro = por_parametro
        self.figura, (self.eixo_peso, self.eixo_silhueta) = plt.subplots(1, 2, figsize=(13, 4.3))
        plt.close(self.figura)  # quem exibe é o handle, não o backend inline
        self.saida = display(self.figura, display_id=True)

    def desenhar(self, tabela: pd.DataFrame, rodape: str, cotovelo=None, melhor=None):
        x = tabela.index.to_numpy(dtype=float)
        for eixo in (self.eixo_peso, self.eixo_silhueta):
            eixo.clear()
            eixo.grid(alpha=0.3)
            eixo.set_xlabel(self.rotulo_x)
            if not self.por_parametro:
                eixo.axvspan(*FAIXA_NEGOCIO, color=config.VERDE, alpha=0.12, lw=0)
                eixo.set_xticks(K_VALORES)
                eixo.set_xlim(K_VALORES[0] - 0.3, K_VALORES[-1] + 0.3)
            if cotovelo is not None:
                eixo.axvline(cotovelo, color=config.AZUL_PROFUNDO, ls="--", lw=1.2,
                             label="cotovelo")

        self.eixo_peso.plot(x, tabela["peso"], "o-", color=config.AZUL_COBALTO)
        self.eixo_peso.set_title(f"Peso — {self.rotulo_peso}", fontsize=10)

        self.eixo_silhueta.plot(x, tabela["silhueta"], "o-", color=config.CORAL)
        self.eixo_silhueta.axhline(config.SILHUETA_MINIMA, color="gray", ls=":", lw=1,
                                   label=f"silhueta mínima ({config.SILHUETA_MINIMA})")
        self.eixo_silhueta.set_title(f"Silhueta — {self.rotulo_silhueta}", fontsize=10)
        if melhor is not None:
            self.eixo_silhueta.plot(melhor, tabela.loc[melhor, "silhueta"], "*", ms=18,
                                    color=config.AMARELO, mec="black", label="maior silhueta")

        if self.por_parametro:
            # O eixo vai do maior para o menor parâmetro: k cresce da esquerda para a direita.
            for eixo in (self.eixo_peso, self.eixo_silhueta):
                eixo.invert_xaxis()
            self.eixo_peso.margins(y=0.25)  # espaço para a anotação acima de cada ponto
            ruido = tabela["pct_ruido"] if "pct_ruido" in tabela else [None] * len(tabela)
            for valor, grupos, peso, pct in zip(x, tabela["grupos"], tabela["peso"], ruido):
                rotulo = f"k={grupos}" if pct is None else f"k={grupos}\n{pct:.0%} ruído"
                self.eixo_peso.annotate(rotulo, (valor, peso),
                                        textcoords="offset points", xytext=(0, 7),
                                        ha="center", fontsize=8)

        self.eixo_silhueta.legend(fontsize=8, loc="best")
        self.figura.suptitle(f"{self.nome} · {rodape}", fontsize=12, fontweight="bold")
        self.figura.tight_layout()
        self.saida.update(self.figura)


def varrer(nome, valores, ajustar, medir_silhueta, rotulo_peso, rotulo_silhueta,
           rotulo_x="k — número de grupos", por_parametro=False, preparo=0.0):
    # Treina `ajustar(valor)` para cada valor, mede peso e silhueta e atualiza o painel.
    #
    # `ajustar` devolve um dicionário com `rotulos`, `peso` e, opcionalmente, colunas
    # extras para a tabela. `preparo` soma ao tempo o custo feito uma vez fora do laço.
    painel = CotoveloAoVivo(nome, rotulo_peso, rotulo_silhueta, rotulo_x, por_parametro)
    linhas = []
    for valor in valores:
        inicio = time.perf_counter()
        saida = dict(ajustar(valor))
        segundos = time.perf_counter() - inicio
        rotulos = saida.pop("rotulos")
        validos = rotulos[rotulos >= 0]
        linhas.append({
            "x": valor,
            "grupos": contar_grupos(rotulos),
            "peso": saida.pop("peso"),
            "silhueta": medir_silhueta(rotulos),
            "silhueta_comum": silhueta_comum(rotulos),
            "maior_grupo_pct": np.bincount(validos).max() / N if len(validos) else np.nan,
            "segundos": segundos,
            **saida,
        })
        tabela = pd.DataFrame(linhas).set_index("x")
        painel.desenhar(tabela, f"{rotulo_x.split(' ')[0]} = {valor} treinado em {segundos:.1f}s")

    # O cotovelo só existe numa curva com pelo menos três pontos distintos.
    # `part.cotovelo` devolve um k inteiro; com o eixo no parâmetro (contínuo), a versão
    # contínua do mesmo critério devolve o valor do parâmetro.
    cotovelo = None
    if tabela["peso"].nunique() >= 3:
        medir_cotovelo = cotovelo_continuo if por_parametro else part.cotovelo
        cotovelo = medir_cotovelo(tabela["peso"])
    melhor = tabela["silhueta"].idxmax() if tabela["silhueta"].notna().any() else None
    total = tabela["segundos"].sum() + preparo
    painel.desenhar(tabela, f"varredura concluída em {total:.1f}s", cotovelo, melhor)

    RESULTADOS[nome] = {"tabela": tabela, "cotovelo": cotovelo, "melhor": melhor,
                        "segundos": total, "por_parametro": por_parametro}
    tabela.index.name = rotulo_x.split(" ")[0]
    display(tabela.style.format(precision=4, na_rep="—").format(
        {"peso": "{:,.2f}", "maior_grupo_pct": "{:.1%}", "segundos": "{:.2f}"}
    ))
    return tabela

# 4 · Os algoritmos

## 4.1 · K-Means — distância euclidiana

O K-Means clássico: cada imóvel vai para o centroide mais próximo e cada centroide vira a
média do seu grupo, até estabilizar. O peso é a **inércia**, a soma dos quadrados das
distâncias ao centroide. Dez inicializações `k-means++` por *k*; fica a de menor inércia.

In [ ]:
def ajustar_kmeans_euclidiano(k):
    modelo = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=SEMENTE).fit(X_T)
    return {"rotulos": modelo.labels_, "peso": modelo.inertia_}


kmeans_euclidiano = varrer(
    "K-Means · euclidiana", K_VALORES, ajustar_kmeans_euclidiano,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="inércia (soma dos quadrados)", rotulo_silhueta="euclidiana",
)

## 4.2 · K-Means — distância Manhattan

Troca-se a régua: o imóvel vai para o centro com menor soma de diferenças absolutas
(distância L1). Com essa régua, a média **deixa de ser** o melhor centro — quem minimiza a
soma das distâncias L1 é a **mediana**, coordenada a coordenada. Manter a média faria o
peso subir em algumas iterações e o algoritmo não convergiria. Por isso a literatura chama
esta variante de *K-Medianas*: é o K-Means com a régua Manhattan levada às últimas
consequências.

O `scikit-learn` não traz essa variante; a implementação abaixo é o laço de Lloyd com a
atribuição em L1 e a atualização pela mediana, com inicialização no estilo `k-means++`.
O peso é a soma das distâncias L1 de cada imóvel ao seu centro.

In [ ]:
def kmeans_manhattan(dados, k, reinicios=10, max_iteracoes=100, semente=SEMENTE):
    # Laço de Lloyd com distância L1: atribuição ao centro mais próximo, centro = mediana.
    gerador = np.random.default_rng(semente)
    n = len(dados)
    melhor = None
    for _ in range(reinicios):
        # Inicialização k-means++ com a régua L1: cada centro novo sorteado longe dos atuais.
        centros = [dados[gerador.integers(n)]]
        for _ in range(1, k):
            proximidade = cdist(dados, np.array(centros), "cityblock").min(axis=1)
            centros.append(dados[gerador.choice(n, p=proximidade / proximidade.sum())])
        centros = np.array(centros)

        for _ in range(max_iteracoes):
            rotulos = cdist(dados, centros, "cityblock").argmin(axis=1)
            novos = np.array([
                np.median(dados[rotulos == g], axis=0) if np.any(rotulos == g) else centros[g]
                for g in range(k)
            ])
            if np.allclose(novos, centros):
                break
            centros = novos

        distancias = cdist(dados, centros, "cityblock")
        rotulos = distancias.argmin(axis=1)
        custo = float(distancias[np.arange(n), rotulos].sum())
        if melhor is None or custo < melhor[1]:
            melhor = (rotulos, custo)
    return melhor


def ajustar_kmeans_manhattan(k):
    rotulos, custo = kmeans_manhattan(X_T, k)
    return {"rotulos": rotulos, "peso": custo}


kmeans_manhattan_tabela = varrer(
    "K-Means · Manhattan", K_VALORES, ajustar_kmeans_manhattan,
    silhueta_sobre(D_MANHATTAN),
    rotulo_peso="soma das distâncias L1", rotulo_silhueta="Manhattan",
)

## 4.3 · K-Means — distância de Mahalanobis

O K-Means euclidiano roda sobre o espaço branqueado do bloco 2, e isso é o K-Means com
Mahalanobis. O peso é a soma dos quadrados das distâncias de Mahalanobis ao centroide, e a
silhueta é medida em Mahalanobis.

O que observar: o branqueamento dá **a mesma variância a toda direção**, inclusive às de
variância mínima — uma nominal quase constante, uma área que só alguns imóveis têm. Essa
direção passa a pesar tanto quanto a área construída. Se a varredura isolar grupos
minúsculos, é esse o motivo — confira a coluna `maior_grupo_pct`.

In [ ]:
def ajustar_kmeans_mahalanobis(k):
    modelo = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=SEMENTE)
    modelo.fit(Z_MAHALANOBIS)
    return {"rotulos": modelo.labels_, "peso": modelo.inertia_}


kmeans_mahalanobis = varrer(
    "K-Means · Mahalanobis", K_VALORES, ajustar_kmeans_mahalanobis,
    silhueta_sobre(D_MAHALANOBIS),
    rotulo_peso="soma dos quadrados de Mahalanobis", rotulo_silhueta="Mahalanobis",
)

## 4.4 · K-Medoids

O centro do grupo deixa de ser uma média e passa a ser um **imóvel real da carteira**, o
medoide: o que tem a menor soma de distâncias aos demais do grupo. É a implementação do
projeto (`part.kmedoids`, iteração de Voronoi com inicialização `k-means++`), sobre a
matriz euclidiana — a mesma régua do 4.1, para que a diferença entre os dois venha só do
centro. O peso é a soma das distâncias (não ao quadrado) de cada imóvel ao seu medoide.

In [ ]:
def ajustar_kmedoids(k):
    resultado = part.kmedoids(D_EUCLIDIANA, k, reinicios=5, semente=SEMENTE)
    return {"rotulos": resultado.rotulos, "peso": resultado.custo}


kmedoids = varrer(
    "K-Medoids", K_VALORES, ajustar_kmedoids,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma das distâncias ao medoide", rotulo_silhueta="euclidiana",
)

## 4.5 · Modelos de Mistura Gaussiana (GMM)

O GMM supõe que a carteira é a soma de *k* gaussianas e estima, para cada imóvel, a
probabilidade de pertencer a cada uma. O rótulo é a componente mais provável. O peso é a
**−log-verossimilhança** total: quanto menor, melhor a mistura explica os dados — e, como a
inércia, ela cai a cada componente acrescentada. A tabela traz também o **BIC**, que
penaliza o número de parâmetros e por isso **não** cai sempre: é o contraponto do próprio
GMM ao cotovelo.

Covariância `full` com `reg_covar=1e-3`: a covariância é singular (bloco 2), e a
regularização soma uma variância mínima na diagonal para que a gaussiana exista.

In [ ]:
def ajustar_gmm(k):
    modelo = GaussianMixture(
        n_components=k, covariance_type="full", reg_covar=1e-3,
        max_iter=200, random_state=SEMENTE,
    ).fit(X_T)
    return {
        "rotulos": modelo.predict(X_T),
        "peso": -modelo.score(X_T) * N,
        "bic": modelo.bic(X_T),
    }


gmm = varrer(
    "Mistura gaussiana (GMM)", K_VALORES, ajustar_gmm,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="−log-verossimilhança", rotulo_silhueta="euclidiana",
)

## 4.6 · Clusterização hierárquica (Ward)

A aglomerativa começa com cada imóvel sozinho e funde, a cada passo, o par de grupos que
menos aumenta a soma dos quadrados intra-grupo (critério de Ward). O dendrograma é
construído **uma vez**; cada *k* é um corte nele (`hier.cortar`). O tempo de construção
entra no total da varredura.

O peso é a soma dos quadrados intra-grupo — exatamente o que Ward minimiza a cada fusão.

In [ ]:
inicio = time.perf_counter()
ligacao_ward = hier.matriz_ligacao(X_T, ligacao="ward")
preparo_ward = time.perf_counter() - inicio


def ajustar_hierarquica(k):
    rotulos = hier.cortar(ligacao_ward, k)
    return {"rotulos": rotulos, "peso": soma_quadrados_intra(X_T, rotulos)}


hierarquica = varrer(
    "Hierárquica (Ward)", K_VALORES, ajustar_hierarquica,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma dos quadrados intra-grupo", rotulo_silhueta="euclidiana",
    preparo=preparo_ward,
)

## 4.7 · Mean Shift

O Mean Shift desloca uma janela de raio `bandwidth` em direção à região mais densa até
parar num **modo** da densidade; imóveis que param no mesmo modo formam um grupo. O número
de grupos **emerge** da largura da janela: janela larga, poucos modos; janela estreita,
muitos.

A varredura percorre larguras estimadas por quantis da distância entre vizinhos
(`estimate_bandwidth`), da maior para a menor — então *k* cresce da esquerda para a direita
e cada ponto do peso traz o *k* que resultou. O peso é a soma dos quadrados intra-grupo.

`bin_seeding=False`: mesmo com ~80 dimensões a grade de sementes do `bin_seeding` fica vazia
(nenhum imóvel cai a uma largura de banda de alguma semente) e o ajuste falha. Cada imóvel
vira semente; é mais lento, e funciona.

In [ ]:
QUANTIS_MEAN_SHIFT = (0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.07)
larguras = sorted(
    {round(float(estimate_bandwidth(X_T, quantile=q, n_samples=1000, random_state=SEMENTE)), 3)
     for q in QUANTIS_MEAN_SHIFT},
    reverse=True,
)


def ajustar_mean_shift(largura):
    modelo = MeanShift(bandwidth=largura, bin_seeding=False, n_jobs=-1).fit(X_T)
    return {"rotulos": modelo.labels_, "peso": soma_quadrados_intra(X_T, modelo.labels_)}


mean_shift = varrer(
    "Mean Shift", larguras, ajustar_mean_shift,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma dos quadrados intra-grupo", rotulo_silhueta="euclidiana",
    rotulo_x="bandwidth — largura da janela", por_parametro=True,
)

## 4.8 · DBSCAN

O DBSCAN junta imóveis que têm pelo menos `min_samples` vizinhos num raio `eps`; o que não
alcança densidade suficiente vira **ruído** (rótulo −1). Como no Mean Shift, *k* emerge do
parâmetro.

O DBSCAN tem um cotovelo próprio, anterior à varredura: a **curva k-distância** — a
distância de cada imóvel ao seu `min_samples`-ésimo vizinho, ordenada. O joelho dela é o
`eps` a partir do qual os pontos deixam de ter vizinhos próximos. Ela vem primeiro, e a
varredura de `eps` é feita em percentis dessa curva, do maior para o menor raio.

O peso precisa decidir o que fazer com o ruído. Ignorá-lo faria o peso cair só porque os
imóveis estão sendo **descartados**, não agrupados. Aqui o ruído paga o custo de quem não
tem grupo: a distância ao centro da carteira inteira, a mesma que pagaria com *k* = 1. A
silhueta é medida sem o ruído, e a coluna `pct_ruido` mostra quanto ficou de fora.

In [ ]:
MIN_AMOSTRAS = 10

curva = den.curva_k_distancia(X_T, MIN_AMOSTRAS)
indice_joelho, eps_joelho = den.joelho_k_distancia(curva)

figura, eixo = plt.subplots(figsize=(8, 4))
eixo.plot(curva, color=config.AZUL_COBALTO)
eixo.axhline(eps_joelho, color=config.CORAL, ls="--", label=f"joelho: eps = {eps_joelho:.2f}")
eixo.set_xlabel("imóveis, ordenados")
eixo.set_ylabel(f"distância ao {MIN_AMOSTRAS}º vizinho")
eixo.set_title("DBSCAN · curva k-distância", fontweight="bold")
eixo.grid(alpha=0.3)
eixo.legend()
plt.show()

In [ ]:
PERCENTIS_EPS = (95, 90, 80, 70, 60, 50, 40, 30, 20, 10)
valores_eps = [round(float(np.percentile(curva, p)), 3) for p in PERCENTIS_EPS]
centro_da_carteira = X_T.mean(axis=0)


def ajustar_dbscan(eps):
    rotulos = DBSCAN(eps=eps, min_samples=MIN_AMOSTRAS).fit_predict(X_T)
    ruido = rotulos < 0
    # Agrupados pagam a distância ao centro do grupo; ruído, ao centro da carteira.
    peso = soma_quadrados_intra(X_T[~ruido], rotulos[~ruido]) if (~ruido).any() else 0.0
    peso += float(((X_T[ruido] - centro_da_carteira) ** 2).sum())
    return {"rotulos": rotulos, "peso": peso, "pct_ruido": ruido.mean()}


dbscan = varrer(
    "DBSCAN", valores_eps, ajustar_dbscan,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma dos quadrados, ruído sem grupo",
    rotulo_silhueta="euclidiana, sem ruído",
    rotulo_x="eps — raio de vizinhança", por_parametro=True,
)

# 5 · O comparativo

Para cada algoritmo, dois candidatos a *k*: o do **cotovelo** do peso e o da **maior
silhueta**. A silhueta de cada um está na régua do próprio algoritmo; a **silhueta comum**
é a euclidiana, a mesma para todos, e é ela que permite ler a tabela na vertical.

Duas colunas protegem a leitura: `maior grupo` e `ruído`. Uma silhueta alta com um grupo
que concentra quase toda a carteira não é segmentação — é a carteira inteira mais um punhado
de exceções.

In [ ]:
def linha_do_resumo(nome, r):
    tabela, cotovelo, melhor = r["tabela"], r["cotovelo"], r["melhor"]

    def em(x, coluna):
        return tabela.loc[x, coluna] if x is not None else np.nan

    na_faixa = tabela[tabela["grupos"].between(*FAIXA_NEGOCIO)]["silhueta"].dropna()
    melhor_faixa = na_faixa.idxmax() if len(na_faixa) else None
    return {
        "algoritmo": nome,
        "k no cotovelo": em(cotovelo, "grupos"),
        "silhueta no cotovelo": em(cotovelo, "silhueta"),
        "k de maior silhueta": em(melhor, "grupos"),
        "maior silhueta": em(melhor, "silhueta"),
        "silhueta comum": em(melhor, "silhueta_comum"),
        "maior grupo": em(melhor, "maior_grupo_pct"),
        "ruído": em(melhor, "pct_ruido") if "pct_ruido" in tabela else 0.0,
        "k de maior silhueta em 2–5": em(melhor_faixa, "grupos"),
        "silhueta comum em 2–5": em(melhor_faixa, "silhueta_comum"),
        "tempo (s)": r["segundos"],
    }


resumo = pd.DataFrame([linha_do_resumo(n, r) for n, r in RESULTADOS.items()]).set_index("algoritmo")
for coluna in ("k no cotovelo", "k de maior silhueta", "k de maior silhueta em 2–5"):
    resumo[coluna] = resumo[coluna].astype("Int64")

display(
    resumo.style
    .format(precision=3, na_rep="—")
    .format({"maior grupo": "{:.1%}", "ruído": "{:.1%}", "tempo (s)": "{:.1f}"}, na_rep="—")
    .highlight_max(subset=["silhueta comum", "silhueta comum em 2–5"], color="#d8f3dc")
)

In [ ]:
figura, (eixo_peso, eixo_silhueta) = plt.subplots(1, 2, figsize=(14, 5))
cores = plt.cm.tab10.colors

for cor, (nome, r) in zip(cores, RESULTADOS.items()):
    tabela = r["tabela"].sort_values("grupos")
    estilo = "s--" if r["por_parametro"] else "o-"
    # Peso normalizado: 1 no maior peso de cada algoritmo, 0 no menor. Os pesos têm
    # unidades diferentes (e a −log-verossimilhança do GMM pode ser negativa).
    peso = tabela["peso"]
    eixo_peso.plot(tabela["grupos"], (peso - peso.min()) / (peso.max() - peso.min()), estilo,
                   color=cor, label=nome, alpha=0.85)
    eixo_silhueta.plot(tabela["grupos"], tabela["silhueta_comum"], estilo,
                       color=cor, label=nome, alpha=0.85)

for eixo in (eixo_peso, eixo_silhueta):
    eixo.axvspan(*FAIXA_NEGOCIO, color=config.VERDE, alpha=0.12, lw=0)
    eixo.set_xlabel("k — grupos obtidos")
    eixo.set_xlim(0.5, K_VALORES[-1] + 0.5)
    eixo.grid(alpha=0.3)
eixo_silhueta.axhline(config.SILHUETA_MINIMA, color="gray", ls=":", lw=1)
eixo_peso.set_title("Peso normalizado (1 = maior, 0 = menor de cada algoritmo)", fontsize=10)
eixo_silhueta.set_title("Silhueta comum (euclidiana, mesmo espaço)", fontsize=10)
eixo_silhueta.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
figura.suptitle("Todos os algoritmos na mesma régua", fontsize=12, fontweight="bold")
figura.tight_layout()
plt.show()

## 6 · A leitura

A regra para escolher, declarada antes de olhar o resultado:

1. o *k* precisa estar na faixa que o negócio opera (2 a 5 grupos);
2. o ruído não pode passar do teto do projeto, e nenhum grupo pode concentrar mais de 90%
   da carteira — acima disso a partição não segmenta, separa exceções;
3. entre os que sobram, vence a maior **silhueta comum**; abaixo da silhueta mínima do
   projeto, o vencedor é registrado com a ressalva de que a estrutura é fraca.

A célula abaixo aplica a regra sobre as varreduras e diz o que sobrou.

In [ ]:
GRUPO_DOMINANTE_MAXIMO = 0.90

candidatos = pd.concat(
    [r["tabela"].assign(algoritmo=nome).reset_index(drop=True) for nome, r in RESULTADOS.items()],
    ignore_index=True,
)
if "pct_ruido" not in candidatos:
    candidatos["pct_ruido"] = np.nan
candidatos["pct_ruido"] = candidatos["pct_ruido"].fillna(0.0)

aprovados = candidatos[
    candidatos["grupos"].between(*FAIXA_NEGOCIO)
    & (candidatos["pct_ruido"] <= config.RUIDO_MAXIMO)
    & (candidatos["maior_grupo_pct"] <= GRUPO_DOMINANTE_MAXIMO)
    & candidatos["silhueta_comum"].notna()
].sort_values("silhueta_comum", ascending=False)

reprovados = sorted(set(RESULTADOS) - set(aprovados["algoritmo"]))
display(Markdown(
    f"**{len(aprovados)}** configurações passam na regra, de **{len(candidatos)}** treinadas. "
    + (f"Sem nenhuma aprovada: {', '.join(reprovados)}." if reprovados else "")
))

if len(aprovados):
    display(
        aprovados[["algoritmo", "grupos", "silhueta", "silhueta_comum", "maior_grupo_pct",
                   "pct_ruido"]]
        .head(10).style.hide(axis="index")
        .format(precision=3).format({"maior_grupo_pct": "{:.1%}", "pct_ruido": "{:.1%}"})
    )
    vencedor = aprovados.iloc[0]
    ressalva = (
        "" if vencedor["silhueta_comum"] >= config.SILHUETA_MINIMA
        else f" — **abaixo da silhueta mínima do projeto ({config.SILHUETA_MINIMA})**: "
        "a estrutura existe, mas é fraca"
    )
    display(Markdown(
        f"Maior silhueta comum dentro da regra: **{vencedor['algoritmo']}** com "
        f"**k = {vencedor['grupos']}** (silhueta comum **{vencedor['silhueta_comum']:.3f}**)"
        f"{ressalva}."
    ))

# 7 · O que melhorou a silhueta — o *label encoding* ou o preço?

As varreduras acima mudaram **duas** coisas ao mesmo tempo em relação ao 04.1. Para saber
qual delas mexeu na silhueta, os mesmos algoritmos rodam em três espaços, com a mesma
semente:

| Espaço | Nominais | Preço | Papel |
|---|---|---|---|
| one-hot, sem preço | *one-hot* | fora | o espaço do 04.1, a linha de base |
| label, sem preço | *label encoding* | fora | isola o efeito do *label encoding* |
| label, com preço | *label encoding* | dentro | o espaço deste notebook; a diferença para o anterior é **só o preço** |

Entram os algoritmos de centro, que respondem a *k* diretamente e rodam em segundos:
K-Means euclidiano, K-Medoids e Ward. A silhueta é a euclidiana de cada espaço.

O ARI (*adjusted Rand index*) completa a leitura: compara, imóvel a imóvel, a partição com
e sem o preço. 1 quer dizer os mesmos grupos; perto de 0, grupos sem relação.

In [ ]:
ESPACOS = {
    "one-hot, sem preço (04.1)": montar_preprocessador("one-hot", com_preco=False),
    "label, sem preço": montar_preprocessador("label", com_preco=False),
    "label, com preço (04.2)": montar_preprocessador("label", com_preco=True),
}
K_ABLACAO = range(2, 11)

linhas = []
particoes = {}
for nome_espaco, preparador in ESPACOS.items():
    inicio = time.perf_counter()
    dados = preparador.fit_transform(X)
    distancias = squareform(pdist(dados, "euclidean"))
    ligacao = hier.matriz_ligacao(dados, ligacao="ward")
    for k in K_ABLACAO:
        rotulos_por_algoritmo = {
            "K-Means": KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit_predict(dados),
            "K-Medoids": part.kmedoids(distancias, k, reinicios=5, semente=SEMENTE).rotulos,
            "Ward": hier.cortar(ligacao, k),
        }
        for algoritmo, rotulos in rotulos_por_algoritmo.items():
            particoes[(nome_espaco, algoritmo, k)] = rotulos
            linhas.append({
                "espaço": nome_espaco, "algoritmo": algoritmo, "k": k,
                "colunas": dados.shape[1],
                "silhueta": silhouette_score(distancias, rotulos, metric="precomputed"),
            })
    print(f"{nome_espaco}: {dados.shape[1]} colunas, {time.perf_counter() - inicio:.1f}s")

ablacao = pd.DataFrame(linhas)

In [ ]:
figura, eixos = plt.subplots(1, 3, figsize=(15, 4.3), sharey=True)
estilos = {
    "one-hot, sem preço (04.1)": ("o:", "gray"),
    "label, sem preço": ("s--", config.AZUL_CLARO),
    "label, com preço (04.2)": ("o-", config.CORAL),
}
for eixo, (algoritmo, grupo) in zip(eixos, ablacao.groupby("algoritmo", sort=False)):
    for nome_espaco, linha in grupo.groupby("espaço", sort=False):
        marcador, cor = estilos[nome_espaco]
        eixo.plot(linha["k"], linha["silhueta"], marcador, color=cor, label=nome_espaco)
    eixo.axvspan(*FAIXA_NEGOCIO, color=config.VERDE, alpha=0.12, lw=0)
    eixo.axhline(config.SILHUETA_MINIMA, color="gray", ls=":", lw=1)
    eixo.set_title(algoritmo, fontsize=11)
    eixo.set_xlabel("k — número de grupos")
    eixo.set_xticks(list(K_ABLACAO))
eixos[0].set_ylabel("silhueta (euclidiana do próprio espaço)")
eixos[-1].legend(fontsize=8, loc="upper right")
figura.suptitle("O mesmo algoritmo em três espaços", fontsize=12, fontweight="bold")
figura.tight_layout()
plt.show()

In [ ]:
# Silhueta por espaço, na faixa que o negócio opera (k de 2 a 5).
na_faixa = ablacao[ablacao["k"].between(*FAIXA_NEGOCIO)]
tabela_espacos = na_faixa.pivot_table(
    index=["algoritmo", "k"], columns="espaço", values="silhueta", sort=False
)[list(ESPACOS)]
tabela_espacos["efeito do label encoding"] = (
    tabela_espacos["label, sem preço"] - tabela_espacos["one-hot, sem preço (04.1)"]
)
tabela_espacos["efeito do preço"] = (
    tabela_espacos["label, com preço (04.2)"] - tabela_espacos["label, sem preço"]
)

# Quanto o preço mexe nos grupos: ARI entre as partições com e sem o preço.
tabela_espacos["ARI com × sem preço"] = [
    adjusted_rand_score(
        particoes[("label, sem preço", algoritmo, k)],
        particoes[("label, com preço (04.2)", algoritmo, k)],
    )
    for algoritmo, k in tabela_espacos.index
]

display(
    tabela_espacos.style.format("{:.3f}")
    .format("{:+.3f}", subset=["efeito do label encoding", "efeito do preço"])
    .background_gradient(cmap="RdYlGn", vmin=-0.05, vmax=0.05,
                         subset=["efeito do label encoding", "efeito do preço"])
)

In [ ]:
efeito_label = tabela_espacos["efeito do label encoding"]
efeito_preco = tabela_espacos["efeito do preço"]
melhor_label_com_preco = tabela_espacos["label, com preço (04.2)"].max()
# Na média sobre todos os pares, a distância quadrática soma 2·var(coluna) de cada coluna:
# a fatia do preço é a variância dele sobre a soma das variâncias.
nomes_das_colunas = list(preprocessor.get_feature_names_out())
variancias = X_T.var(axis=0)
peso_do_preco = variancias[nomes_das_colunas.index("continuous__preco_venda")] / variancias.sum()

display(Markdown(
    f"- **label encoding**: muda a silhueta entre **{efeito_label.min():+.3f}** e "
    f"**{efeito_label.max():+.3f}** (média **{efeito_label.mean():+.3f}**) "
    "na faixa de 2 a 5 grupos;\n"
    f"- **preço**: muda a silhueta entre **{efeito_preco.min():+.3f}** e "
    f"**{efeito_preco.max():+.3f}** (média **{efeito_preco.mean():+.3f}**), e os grupos com e "
    f"sem ele concordam em ARI médio de **{tabela_espacos['ARI com × sem preço'].mean():.3f}**;\n"
    f"- maior silhueta do espaço deste notebook na faixa de negócio: "
    f"**{melhor_label_com_preco:.3f}** — "
    + ("acima" if melhor_label_com_preco >= config.SILHUETA_MINIMA else "abaixo")
    + f" da silhueta mínima do projeto ({config.SILHUETA_MINIMA});\n"
    f"- o preço é **1 de {len(nomes_das_colunas)} colunas** e responde por "
    f"**{peso_do_preco:.1%}** da distância quadrática média entre dois imóveis."
))

## Como ler o que saiu

* **O peso cai sempre, e por isso sozinho não escolhe nada.** Com *k* = *n* o peso é zero.
  O cotovelo só diz onde a queda perde fôlego; se a curva desce em linha quase reta, não há
  cotovelo de verdade e o ponto marcado é o menos ruim da corda.
* **O *label encoding* melhora a silhueta porque encolhe o espaço, não porque descobre
  grupos.** Saem ~130 colunas 0/1 que deixavam todos os imóveis à mesma distância uns dos
  outros. O preço do ganho é uma ordem alfabética entre categorias, que nenhum segmento
  consegue explicar a quem opera a carteira.
* **Na silhueta, o preço quase não pesa.** É uma coluna entre ~80, na mesma escala de [0, 1]
  das outras, e responde por uma fatia pequena da distância (bloco 7). Para ele mover a
  silhueta, teria de ganhar peso — e aí a segmentação vira faixa de preço, que a carteira já
  tem sem precisar de modelo.
* **Nos grupos, o efeito do preço depende do algoritmo.** No K-Means as partições com e sem
  ele praticamente coincidem. A hierárquica decide as fusões de forma gulosa: uma coluna a
  mais pode mudar o caminho do dendrograma e trocar muitos imóveis de grupo sem mudar a
  silhueta. A coluna de ARI do bloco 7 mostra onde isso acontece.
* **Silhueta baixa em todos os algoritmos continua sendo um resultado, não um defeito.** Com
  ~80 dimensões as distâncias ainda se concentram: os imóveis variam de forma contínua, sem
  vazios entre grupos. É o mesmo diagnóstico do notebook 04.
* **Mean Shift e DBSCAN dependem de densidade**, e densidade é o que mais se dilui em alta
  dimensão. É esperado que formem um grupo gigante com satélites, ou que mandem boa parte da
  carteira para o ruído. A curva k-distância sem joelho nítido já antecipa isso.
* **Mahalanobis amplifica o raro.** Ao igualar a variância de toda direção, ele dá a uma
  categoria com poucos imóveis o mesmo peso da área construída — e tende a isolar exatamente
  esses imóveis.
* **Na régua comum, as variantes de centro (K-Means, K-Medoids, Ward, GMM) costumam ficar
  próximas.** Quando isso acontece, o desempate é de negócio: o medoide é um imóvel real,
  o GMM dá grau de pertencimento, o Ward dá o dendrograma para justificar o corte.